In [2]:
import os

def encoding_report(text, encodings):
    """Prints a table: encoding, bytes, first 8 bytes in hex."""
    print(f"{'Encoding':<10} {'Bytes':>6}  Beginning")
    print("-" * 42)
    for enc in encodings:
        try:
            data = text.encode(enc)
            print(f"{enc:<10} {len(data):>6}  {data[:8].hex(' ')}")
        except UnicodeEncodeError:
            print(f"{enc:<10} {'not supported':>6}")

SAMPLE = ("Станция №7: t=+21.5°С\n"
          "Ветер 3 м/с, влажность 60%\n"
          "Цена: 25 €, датчик #42\n"
          "Hello, World! 12345\n"
          "Тест завершён успешно.\n")

encoding_report(SAMPLE, ["utf-8", "cp1251", "utf-16", "latin-1"])

for enc in ["utf-8", "utf-8-sig", "cp1251", "utf-16"]:
    fname = f"text_{enc.replace('-', '_')}.txt"
    with open(fname, "w", encoding=enc) as f:
        f.write(SAMPLE)
    with open(fname, "rb") as f:
        head = f.read(4)
    size = os.path.getsize(fname)
    bom = ""
    if head.startswith(b"\xef\xbb\xbf"):
        bom = " (BOM)"
    elif head.startswith(b"\xff\xfe") or head.startswith(b"\xfe\xff"):
        bom = " (BOM)"
    print(f"File {fname}: {size} bytes, beginning {head.hex(' ')}{bom}")

with open("text_cp1251.txt", "rb") as f:
    raw = f.read()

print("\n--- strict ---")
try:
    print(raw.decode("utf-8"))
except UnicodeDecodeError as e:
    print(f"UnicodeDecodeError: {e}")

print("\n--- replace ---")
print(raw.decode("utf-8", errors="replace"))

print("\n--- ignore ---")
print(raw.decode("utf-8", errors="ignore"))

def convert_encoding(src, src_enc, dst, dst_enc):
    """Recodes a text file, returns number of characters."""
    with open(src, "r", encoding=src_enc) as f:
        content = f.read()
    with open(dst, "w", encoding=dst_enc) as f:
        f.write(content)
    return len(content)

n_chars = convert_encoding("text_cp1251.txt", "cp1251", "temp_utf8.txt", "utf-8")
print(f"\nCharacters: {n_chars}")
convert_encoding("temp_utf8.txt", "utf-8", "roundtrip_cp1251.txt", "cp1251")

with open("text_cp1251.txt", "rb") as f1:
    orig = f1.read()
with open("roundtrip_cp1251.txt", "rb") as f2:
    rt = f2.read()
print(f"Byte-identical after cp1251 -> UTF-8 -> cp1251: {orig == rt}")

Encoding    Bytes  Beginning
------------------------------------------
utf-8         173  d0 a1 d1 82 d0 b0 d0 bd
cp1251        115  d1 f2 e0 ed f6 e8 ff 20
utf-16        232  ff fe 21 04 42 04 30 04
latin-1    not supported
File text_utf_8.txt: 178 bytes, beginning d0 a1 d1 82
File text_utf_8_sig.txt: 181 bytes, beginning ef bb bf d0 (BOM)
File text_cp1251.txt: 120 bytes, beginning d1 f2 e0 ed
File text_utf_16.txt: 242 bytes, beginning ff fe 21 04 (BOM)

--- strict ---
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xd1 in position 0: invalid continuation byte

--- replace ---
������� �7: t=+21.5��
����� 3 �/�, ��������� 60%
����: 25 �, ������ #42
Hello, World! 12345
���� �������� �������.


--- ignore ---
 7: t=+21.5
 3 /,  60%
: 25 ,  #42
Hello, World! 12345
  .


Characters: 115
Byte-identical after cp1251 -> UTF-8 -> cp1251: True


In [3]:

import glob
"""Deletes all .txt files in the current directory."""
for file in glob.glob("*.txt"):
    try:
        os.remove(file)
        print(f"Deleted: {file}")
    except OSError as e:
        print(f"Error deleting {file}: {e}")



Deleted: roundtrip_cp1251.txt
Deleted: temp_utf8.txt
Deleted: text_cp1251.txt
Deleted: text_utf_16.txt
Deleted: text_utf_8.txt
Deleted: text_utf_8_sig.txt
